# 02 — Extraction and classification

**What you will learn**

- How to move one label set to a completely different domain with no reload,
  and why that works.
- How to steer an ambiguous label with a natural-language description, and when
  it is worth the extra tokens.
- `POST /classify_text`: the two payload shapes, why only one is accepted, and
  what the rejection message tells you.
- How to run several independent classification tasks in a single forward pass.
- Why classification and extraction behave differently under the hood, and what
  that means for how much you should trust each.

**What it assumes you already did**

[01 — Getting started](01-getting-started.ipynb). You should already know what
`BASE_URL` is, that response keys mirror your labels, and that entity values are
bare strings by default.

**Roughly how long**

About 20 minutes.

## Setup

Same helpers as notebook 01. Run and move on.

In [1]:
import json
import os
import time

import requests

# Every notebook in this path reads the same environment variable, so you can
# point the whole series at a different deployment with one export:
#     export GLINER_BASE_URL=http://localhost:8013
BASE_URL = os.environ.get("GLINER_BASE_URL", "http://192.168.1.177:8013")

# The server bounds its own inference at REQUEST_TIMEOUT_SECONDS (default 120)
# and returns 504 when it blows through that. A client timeout slightly above
# the server's means the server always gets to explain itself with a status
# code instead of the client giving up first and leaving you guessing.
TIMEOUT = 130

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def get(path):
    """GET a path and return parsed JSON. Raises on non-2xx."""
    r = session.get(f"{BASE_URL}{path}", timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post(path, payload):
    """POST JSON and return parsed JSON. Raises on non-2xx."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post_raw(path, payload):
    """POST JSON and return (status_code, parsed_body_or_text). Never raises.

    Used whenever the interesting part of the answer IS the status code.
    """
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def show(obj):
    """Pretty-print a JSON-serializable object."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)

BASE_URL = http://192.168.1.177:8013


## Entity extraction across domains

The request contract for `POST /extract_entities`:

| Field | Type | Notes |
|---|---|---|
| `text` | string | non-empty, `<= MAX_TEXT_CHARS` (default 20000) |
| `labels` | list of strings, **or** object mapping label → description | non-empty, `<= MAX_LABELS` (default 256) |

Plus the optional inference options covered in notebook 04.

Below are two requests with nothing in common but the endpoint. Clinical text
with clinical labels, then business text with business labels. No reload
happens between them, no schema was registered, and the second request does not
know the first existed.

This is worth sitting with for a moment, because it changes what a "schema" costs
you. In a fine-tuned pipeline, adding an entity type is a labelling project.
Here it is a string in a list. The tradeoff is that you get no training-set
guarantees in return: nothing has been calibrated on *your* data, so quality is
whatever the model's general language understanding gives you on your domain.
Cheap to try, and cheap to discover it does not work.

In [2]:
clinical = post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": ["medication", "dosage", "symptom", "time"],
})
show(clinical)

{
  "entities": {
    "medication": [
      "ibuprofen"
    ],
    "dosage": [
      "400mg"
    ],
    "symptom": [
      "severe headache"
    ],
    "time": [
      "2 PM"
    ]
  }
}


In [3]:
business = post("/extract_entities", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "labels": ["company", "person", "location", "job_title"],
})
show(business)

{
  "entities": {
    "company": [
      "Apple"
    ],
    "person": [
      "Tim Cook"
    ],
    "location": [
      "Cupertino"
    ],
    "job_title": [
      "CEO"
    ]
  }
}


## Steering a label with a description

Passing `labels` as a JSON *object* instead of a list maps each label to a
natural-language description of what you mean by it.

The mechanism is straightforward once you know how the model works: your label
string is embedded and matched against the text. A one-word label like
`"reference"` is a very thin signal — it could plausibly mean a citation, a
support ticket number, a bank payment reference, or a job reference from a
previous employer, and the embedding has to represent all of those at once. A
description gives the encoder considerably more to work with, and pushes the
label's representation toward the sense you actually want.

**When it is worth it.** Use descriptions when a bare label is genuinely
ambiguous in your domain, when a label is jargon the model has no strong prior
for, or when you are getting recall on the wrong sense of a word. Do not reach
for it reflexively: `"person"` and `"company"` are already unambiguous, and a
description there is extra tokens for no gain.

**What it is not.** This is not a rule, a regex, or a constraint. It is a
steering signal, and the model can still ignore it. If you need a hard
guarantee about permitted values, notebook 03 covers `choices`, which is the
closest thing to a constraint this API offers.

In [4]:
described = post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": {
        "medication": "name of a drug administered to the patient",
        "dosage": "amount and unit of the drug",
    },
})
show(described)

{
  "entities": {
    "medication": [
      "ibuprofen"
    ],
    "dosage": [
      "400mg"
    ]
  }
}


Note that the response shape is identical either way — keys are the label names,
values are lists. The description exists only in the request. So switching a
label from bare to described is a safe change for your parsing code, which makes
it easy to iterate on: try a description, compare output, keep or discard.

### Multilingual, and the span-boundary warning

`fastino/gliner2.5-multi-v1` is the multilingual checkpoint, built on an
mDeBERTa-v3-base encoder. The cell below only produces meaningful results if
this service is running that `MODEL_ID` — the English-only checkpoints will
return *something* for French text, but not something you should act on.

In [5]:
current_model = get("/health")["model_id"]
print("serving:", current_model)

if "multi" not in current_model:
    print("NOTE: not a multilingual checkpoint - treat the output below as a curiosity.")

show(post("/extract_entities", {
    "text": "Luca de Meo, PDG de Renault, a annonce une nouvelle usine a Douai.",
    "labels": ["person", "company", "location"],
}))

serving: fastino/gliner2.5-base-v1
NOTE: not a multilingual checkpoint - treat the output below as a curiosity.


{
  "entities": {
    "person": [
      "Luca de Meo"
    ],
    "company": [
      "Renault"
    ],
    "location": [
      "Douai"
    ]
  }
}


This is the right place for the most important operational warning about this
model family, because it bites hardest when you switch checkpoints.

The output above came from `gliner2.5-base-v1`, which returns `"Renault"` as the
company — the tight span you would want. But on a four-sentence comparison run
on `jarvita-agx`, `gliner2.5-multi` returned `"PDG de Renault"` for a sentence
like this one: it swept the job title into the span. Same sentence, same label,
different checkpoint, different answer.

**Four sentences is not an evaluation**, and no conclusion about which model is
better follows from it. What does follow is one habit worth adopting: when you
change `MODEL_ID`, re-check span boundaries against your own texts before
assuming downstream string matching still works. If your pipeline joins
extracted company names against a reference table, `"PDG de Renault"` misses
where `"Renault"` hits, and nothing anywhere will raise an error. You will just
get fewer joins.

This is the concrete reason notebook 01 told you to stamp
`/version.model_revision` onto stored rows.

## Classification — `POST /classify_text`

Where extraction pulls spans *out* of the text, classification assigns the whole
text to one of a closed set of labels you supply.

| Field | Type | Notes |
|---|---|---|
| `text` | string | non-empty, `<= MAX_TEXT_CHARS` |
| `labels` | **object** mapping task name → list of class labels | non-empty, `<= MAX_LABELS` |

Note that `labels` here means something structurally different from what it
means on `/extract_entities`, even though it is the same field name. On
`/extract_entities` it is a flat set of things to look for. Here it is a
*mapping from a task name to that task's mutually exclusive classes*. The
outer key names the task; the inner list is the answer space for it.

In [6]:
sentiment = post("/classify_text", {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": {"sentiment": ["positive", "negative", "neutral"]},
})
show(sentiment)

{
  "sentiment": "negative"
}


### Why a bare list is rejected

You might reasonably expect `"labels": ["positive", "negative", "neutral"]` to
work here — it is the shape `/extract_entities` takes, and the classes are
right there. The service returns **400** instead. Run it and read the message.

In [7]:
status, body = post_raw("/classify_text", {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": ["positive", "negative", "neutral"],
})
print("HTTP", status)
show(body)

HTTP 400
{
  "detail": "'labels' must be an object mapping a task name to its labels, e.g. {\"sentiment\": [\"positive\", \"negative\"]}. A bare list is not accepted."
}


The message names the shape it wants rather than just saying "invalid", which is
the pattern throughout this API's error handling — notebook 07 covers the full
set.

But *why* is the bare list refused, when the service could obviously have
defaulted the task name to something like `"classification"`?

**Because the response key is derived from the task name, and there is no good
default for it.** Look at what came back from the successful call: `{"sentiment":
"negative"}`. The key is `sentiment` because you named the task `sentiment`. If a
bare list were accepted, the service would have to invent a key, and every
client would then have to hardcode whatever that invented key happened to be —
coupling your parsing code to an implementation detail rather than to something
you chose.

There is a second, stronger reason, visible in the next cell: the object form is
the only shape that can express **more than one classification task**. A bare
list has nowhere to put a second task. Accepting it would mean supporting two
payload shapes where one is a strict subset of the other in capability, and
where the weaker one produces unpredictable response keys. Refusing it keeps
exactly one way to do this, and that way scales.

This is a small instance of a general principle in this API: it prefers a loud
`400` naming the correct shape over a silently-accepted guess. You will see the
same choice made about unknown payload keys in notebook 04.

### Several tasks, one forward pass

Add more keys to the `labels` object and each becomes an independent
classification head over the same text. The whole thing is still **one** forward
pass — the text is encoded once and scored against both answer spaces.

That is the real reason to prefer this over two HTTP calls. On a single-slot GPU
box, two calls means two trips through the inference semaphore and two encodes
of the same text. One call with two tasks means one of each. Notebook 06 takes
this idea considerably further.

In [8]:
show(post("/classify_text", {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": {
        "sentiment": ["positive", "negative", "neutral"],
        "topic": ["battery", "display", "software", "build quality"],
    },
}))

{
  "sentiment": "negative",
  "topic": "battery"
}


Two properties of that response worth naming:

- **One answer per task, not a list.** Unlike extraction, where a label maps to
  a list of found spans, a classification task maps to a single winning label.
  The classes within one task genuinely *are* treated as competing alternatives,
  which is the opposite of the "labels are not a partition" rule you saw for
  extraction in notebook 01. The partition is per task.
- **Tasks are independent of each other.** `sentiment` and `topic` do not
  compete; only the classes inside each one do.

## Try this yourself

Take a text that is genuinely mixed in sentiment — for example, *"The camera is
outstanding but the battery dies in four hours."* — and classify it with
`["positive", "negative", "neutral"]`.

Then ask the harder question: run the same text through `/extract_entities`
with labels `["praised_feature", "criticized_feature"]`.

Predict which one gives you something more useful before you run it.

In [9]:
MIXED = "The camera is outstanding but the battery dies in four hours."

print("--- classification (one answer for the whole text) ---")
show(post("/classify_text", {
    "text": MIXED,
    "labels": {"sentiment": ["positive", "negative", "neutral"]},
}))

print("\n--- extraction (per-span, several answers) ---")
show(post("/extract_entities", {
    "text": MIXED,
    "labels": ["praised_feature", "criticized_feature"],
}))

--- classification (one answer for the whole text) ---


{
  "sentiment": "negative"
}

--- extraction (per-span, several answers) ---


{
  "entities": {
    "praised_feature": [
      "camera"
    ],
    "criticized_feature": [
      "battery"
    ]
  }
}


**Discussion.** The classifier has to pick one label for the entire document, so
whatever it returns for genuinely mixed text is a lossy summary. It is not wrong
exactly — it is answering the question you asked, which was "what is the
sentiment of this text", a question that has no good single answer here.

The extraction framing gives you something structurally richer: it tells you
*which part* of the text was praised and which was criticized, so you keep the
mixedness instead of collapsing it. For aspect-level opinion work that is
usually what you actually wanted.

The generalizable lesson: **choose the route by the shape of the answer you
need, not by the vocabulary of the task.** "Sentiment analysis" sounds like a
classification problem, and for a short uniform review it is. For anything
longer or more nuanced, per-span extraction or the structured extraction in
notebook 03 gives you a representation you can still reason about afterwards.

A pragmatic middle path: run both in the same request. Notebook 03's
`/extract_multitask` does exactly that — entities, a classification, and a
structured record from one forward pass.

## What you learned

- The same resident weights serve any label set; changing domains costs a list
  of strings, not a training run. In exchange you get no calibration on your
  data.
- Passing `labels` as an object maps each label to a description, which steers
  an ambiguous label toward the sense you mean. It is a signal, not a
  constraint, and the response shape is unchanged.
- Span boundaries are the model's choice and they move between checkpoints
  (`"PDG de Renault"` vs `"Renault"` on a four-sentence sample). Re-check them
  against your own texts whenever `MODEL_ID` changes.
- `/classify_text` requires `labels` as an object mapping task name to classes.
  A bare list is a **400** with a message naming the right shape — because the
  task name determines the response key, and because only the object form can
  carry more than one task.
- Several classification tasks in one request is one forward pass. Classes
  compete within a task; tasks do not compete with each other.
- Classification collapses a document to one label. Extraction keeps the
  structure. Pick by the answer shape you need.

## Next

**[03 — Structured extraction and multi-task](03-structured-and-multitask.ipynb)** —
declaring record schemas, and getting entities, a classification and a record
from a single forward pass.